# 02 · Gap Analysis Agent (RAG)

**Goal:** Compare a user's extracted `ResumeProfile` against the skill requirements for a target role and produce a **ranked list of skill gaps** with prerequisite ordering.

This notebook implements **Stages 3-5** from `Documentation/SkillGraph_ Resume Extraction and RAG Pipeline.pdf`:

1. **Embed the role-skill taxonomy** into a vector store (one node per skill, prerequisites as structured metadata).
2. **Three-step retrieval pipeline**:
   - Step 1 — metadata pre-filter (`role=target_role`, `corpus=taxonomy`)
   - Step 2 — hybrid retrieval: BM25 (lexical) + dense embedding, merged via reciprocal rank fusion
   - Step 3 — cross-encoder reranker (`ms-marco-MiniLM-L-6-v2`) → top-5
3. **Gap identification** with Gemini over the top-5 candidates, producing a structured `GapReport` with prerequisite ordering.
4. Wrap as a **LangGraph subgraph**.

### Pinecone vs. local
Backend docs call for Pinecone. This notebook runs on **local in-memory FAISS + rank_bm25** so you can iterate without Pinecone credentials. Clearly marked `# PRODUCTION` comments show where to swap in Pinecone + metadata filters.

### Why hybrid + reranker?
Per the docs, a single-step semantic search fails on two problems: (a) skill name disambiguation (PyTorch ≠ TensorFlow even though they embed close) — BM25 handles this, and (b) loose relevance: "send Gemini 20 loosely-relevant nodes and it writes meandering gaps; send it 5 high-precision gaps and output is clean and actionable." Cross-encoder rerank is what gets us from 20 → 5.


## 0. Setup

In [ ]:
# !pip install -q langchain langchain-google-genai langchain-core langgraph \
#   sentence-transformers rank_bm25 faiss-cpu pydantic python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path("../backend/.env").resolve())
assert os.getenv("GOOGLE_API_KEY")
print("OK")

## 1. Toy knowledge base

In production each record comes from `sql/` seed files and gets upserted to Pinecone. Here we inline a small taxonomy for the `ml-engineer` role so the notebook runs standalone.

Schema per node:
- `skill`: canonical name
- `description`: short definition
- `role`: which role this skill belongs to
- `skill_category`: Languages / ML / Cloud & DevOps / ...
- `level_required`: beginner / intermediate / advanced
- `prerequisites`: list of skill names (structured, NOT embedded — we traverse in Python)
- `corpus`: "taxonomy" / "course" / "jd"


In [ ]:
TAXONOMY = [
    # Foundations
    {"skill": "Python",           "description": "General-purpose programming language; core language for ML engineering.",
     "role": "ml-engineer", "skill_category": "Languages", "level_required": "intermediate", "prerequisites": []},
    {"skill": "SQL",              "description": "Querying relational databases; essential for data engineering and analytics.",
     "role": "ml-engineer", "skill_category": "Languages", "level_required": "intermediate", "prerequisites": []},
    {"skill": "Linear Algebra",   "description": "Vectors, matrices, eigen-decomposition — foundation for ML math.",
     "role": "ml-engineer", "skill_category": "Math", "level_required": "intermediate", "prerequisites": []},
    {"skill": "Statistics",       "description": "Probability, distributions, hypothesis testing; underpins modeling choices.",
     "role": "ml-engineer", "skill_category": "Math", "level_required": "intermediate", "prerequisites": []},

    # Core ML
    {"skill": "PyTorch",          "description": "Deep learning framework; dynamic graphs, autograd, widely used in research and prod.",
     "role": "ml-engineer", "skill_category": "ML", "level_required": "advanced", "prerequisites": ["Python", "Linear Algebra"]},
    {"skill": "TensorFlow",       "description": "Alternative deep learning framework; static/dynamic graphs; common in Google stack.",
     "role": "ml-engineer", "skill_category": "ML", "level_required": "intermediate", "prerequisites": ["Python", "Linear Algebra"]},
    {"skill": "scikit-learn",     "description": "Classical ML algorithms: regressions, trees, clustering; feature-engineering pipelines.",
     "role": "ml-engineer", "skill_category": "ML", "level_required": "intermediate", "prerequisites": ["Python", "Statistics"]},
    {"skill": "Transformers",     "description": "Attention-based architectures: BERT, GPT; the backbone of modern NLP/LLMs.",
     "role": "ml-engineer", "skill_category": "ML", "level_required": "advanced", "prerequisites": ["PyTorch"]},
    {"skill": "LLM Fine-tuning",  "description": "LoRA/QLoRA/full-finetune on foundation models; evaluation and data prep.",
     "role": "ml-engineer", "skill_category": "ML", "level_required": "advanced", "prerequisites": ["Transformers", "PyTorch"]},

    # Data / infra
    {"skill": "pandas",           "description": "Tabular data manipulation in Python.",
     "role": "ml-engineer", "skill_category": "Data", "level_required": "intermediate", "prerequisites": ["Python"]},
    {"skill": "Airflow",          "description": "DAG-based workflow orchestration for data/ML pipelines.",
     "role": "ml-engineer", "skill_category": "Data", "level_required": "intermediate", "prerequisites": ["Python", "SQL"]},
    {"skill": "MLflow",           "description": "Experiment tracking + model registry + deployment. Standard in MLOps.",
     "role": "ml-engineer", "skill_category": "MLOps", "level_required": "intermediate", "prerequisites": ["Python"]},

    # Cloud / DevOps
    {"skill": "Docker",           "description": "Containerize training and inference workloads for reproducible deployment.",
     "role": "ml-engineer", "skill_category": "Cloud & DevOps", "level_required": "intermediate", "prerequisites": []},
    {"skill": "Kubernetes",       "description": "Container orchestration; scale inference services; run distributed training.",
     "role": "ml-engineer", "skill_category": "Cloud & DevOps", "level_required": "advanced", "prerequisites": ["Docker"]},
    {"skill": "AWS",              "description": "Cloud fundamentals: S3, EC2, Lambda, SageMaker.",
     "role": "ml-engineer", "skill_category": "Cloud & DevOps", "level_required": "intermediate", "prerequisites": []},
    {"skill": "Vector Databases", "description": "Pinecone / Weaviate / FAISS for semantic retrieval in RAG and recommendations.",
     "role": "ml-engineer", "skill_category": "ML", "level_required": "intermediate", "prerequisites": ["Python"]},
    {"skill": "RAG Systems",      "description": "Retrieval-Augmented Generation: chunking, embedding, hybrid retrieval, reranking.",
     "role": "ml-engineer", "skill_category": "ML", "level_required": "advanced", "prerequisites": ["Vector Databases", "Transformers"]},
]

# Mark the corpus for every record
for t in TAXONOMY:
    t["corpus"] = "taxonomy"

print(f"{len(TAXONOMY)} skill nodes loaded")

## 2. Embed the taxonomy

We use **Google `text-embedding-004`** (768-dim, free via AI Studio). Per the docs, use `RETRIEVAL_DOCUMENT` when embedding the corpus and `RETRIEVAL_QUERY` when embedding queries — the model is fine-tuned differently for each and using the wrong type measurably hurts recall.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import numpy as np

embed_docs = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    task_type="RETRIEVAL_DOCUMENT",
)
embed_query = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    task_type="RETRIEVAL_QUERY",
)

# Embed the description (the skill + one-line definition) for each node.
texts = [f"{t['skill']}: {t['description']}" for t in TAXONOMY]
vectors = np.array(embed_docs.embed_documents(texts), dtype="float32")
print("Corpus vectors:", vectors.shape)

In [ ]:
# In-memory FAISS for cosine similarity. PRODUCTION: replace with Pinecone upsert+query.
import faiss
index = faiss.IndexFlatIP(vectors.shape[1])
faiss.normalize_L2(vectors)  # cosine via inner product
index.add(vectors)
print("FAISS index size:", index.ntotal)

## 3. BM25 lexical index

Per the docs: BM25 is crucial for exact skill-name matching (PyTorch vs TensorFlow). For skill lookups: `BM25=0.4, dense=0.6`. For JD lookups: `BM25=0.2, dense=0.8`.


In [ ]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts])
print("BM25 ready.")

## 4. Three-step retrieval pipeline

### Step 1 — Metadata pre-filter
Filter the corpus down to `role=target_role` and `corpus=taxonomy` **before** running similarity search. This removes cross-role noise and makes retrieval cheaper.

### Step 2 — Hybrid (BM25 + dense) with reciprocal rank fusion
### Step 3 — Cross-encoder rerank → top-5


In [ ]:
def prefilter(role: str, corpus: str = "taxonomy"):
    """Return (filtered_texts, filtered_metas, original_indices)."""
    out_texts, out_metas, out_idx = [], [], []
    for i, meta in enumerate(TAXONOMY):
        if meta["role"] == role and meta["corpus"] == corpus:
            out_texts.append(texts[i])
            out_metas.append(meta)
            out_idx.append(i)
    return out_texts, out_metas, out_idx

def dense_search(query: str, k: int = 20, restrict_idx: list[int] | None = None):
    qv = np.array(embed_query.embed_query(query), dtype="float32").reshape(1, -1)
    faiss.normalize_L2(qv)
    scores, ids = index.search(qv, index.ntotal)  # search all, filter after
    out = []
    allowed = set(restrict_idx) if restrict_idx is not None else None
    for s, i in zip(scores[0], ids[0]):
        if allowed is None or i in allowed:
            out.append((int(i), float(s)))
        if len(out) >= k:
            break
    return out

def bm25_search(query: str, k: int = 20, restrict_idx: list[int] | None = None):
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(enumerate(scores), key=lambda x: -x[1])
    allowed = set(restrict_idx) if restrict_idx is not None else None
    out = [(i, float(s)) for i, s in ranked if (allowed is None or i in allowed)]
    return out[:k]

def rrf_merge(results_a, results_b, k: int = 60):
    """Reciprocal Rank Fusion. Returns merged (idx, score) list sorted by score desc."""
    rank_a = {i: r for r, (i, _) in enumerate(results_a)}
    rank_b = {i: r for r, (i, _) in enumerate(results_b)}
    all_ids = set(rank_a) | set(rank_b)
    merged = []
    for i in all_ids:
        score = 0.0
        if i in rank_a: score += 1 / (k + rank_a[i])
        if i in rank_b: score += 1 / (k + rank_b[i])
        merged.append((i, score))
    return sorted(merged, key=lambda x: -x[1])

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: list[int], top_n: int = 5):
    pairs = [(query, texts[i]) for i in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return ranked[:top_n]

In [ ]:
def retrieve_for_gap_query(query: str, role: str, top_n: int = 5):
    # Step 1 — metadata pre-filter
    _, _, idx = prefilter(role=role, corpus="taxonomy")
    # Step 2 — hybrid retrieval
    dense = dense_search(query, k=20, restrict_idx=idx)
    lex   = bm25_search(query, k=20, restrict_idx=idx)
    merged = rrf_merge(dense, lex)[:20]
    candidates = [i for i, _ in merged]
    # Step 3 — cross-encoder rerank
    reranked = rerank(query, candidates, top_n=top_n)
    return [(TAXONOMY[i], score) for i, score in reranked]

# Sanity check: query "deep learning for text"
for meta, sc in retrieve_for_gap_query("deep learning for text and transformers", "ml-engineer"):
    print(f"{sc:.3f}  {meta['skill']:20s}  [{meta['level_required']:12s}]")

## 5. Gap identification with Gemini

We have two approaches:

1. **Subtraction approach** — Python set-diff of (role skills) − (user skills) gives raw gaps. Cheap, deterministic, no LLM call.
2. **Retrieval + LLM approach** — For each user-stated goal / missing skill category, run the RAG pipeline, feed the top-5 retrieved skill nodes to Gemini with the user's profile, and let Gemini name the top gaps with reasoning.

For real users, **do both**: subtraction gives the floor, RAG + LLM adds prioritization and prerequisite ordering. Below is the combined flow.


In [ ]:
from typing import List
from pydantic import BaseModel, Field

class Gap(BaseModel):
    skill: str
    category: str
    relevance: int = Field(description="0-100, how relevant to the target role")
    difficulty: str = Field(description="Easy | Medium | Hard")
    prerequisites: List[str] = Field(description="Prereq skills the user must learn first")
    why: str = Field(description="1 sentence — why this gap is blocking")

class GapReport(BaseModel):
    target_role: str
    gaps: List[Gap] = Field(description="Top gaps ordered with prerequisites earliest")

In [ ]:
# Sample user profile (from notebook 01 output)
USER_SKILLS = ["Python", "pandas", "scikit-learn", "SQL", "Docker"]

# 1. Subtraction pass — raw candidate gaps
role_skills = {t["skill"] for t in TAXONOMY if t["role"] == "ml-engineer"}
raw_gaps = sorted(role_skills - set(USER_SKILLS))
print("Raw subtraction gaps:", raw_gaps)

In [ ]:
# 2. RAG pass — for each raw gap, fetch metadata + neighboring context.
# We retrieve using a query built from (target role + raw gap names) to surface the 5 most impactful gaps.
query = f"ml-engineer role missing: {', '.join(raw_gaps)}"
top5 = retrieve_for_gap_query(query, role="ml-engineer", top_n=8)

retrieved_context = "\n".join(
    f"- {m['skill']} (category={m['skill_category']}, level={m['level_required']}, prereqs={m['prerequisites']}): {m['description']}"
    for m, _ in top5
)
print(retrieved_context)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
structured = gemini.with_structured_output(GapReport)

prompt = ChatPromptTemplate.from_template('''You are an expert career coach producing skill-gap analysis.

USER'S CURRENT SKILLS:
{user_skills}

TARGET ROLE: {target_role}

RETRIEVED ROLE-SKILL NODES (these are the top candidates for gaps):
{context}

Select the TOP 5 gaps the user should close, ordered so prerequisites come FIRST.
- Use only skills from the retrieved nodes (don't invent new ones).
- `relevance` is 0-100 for impact on the target role.
- `difficulty` is Easy / Medium / Hard based on the user's current profile.
- `prerequisites` is the list of prereq skills the user must learn first (from the nodes above).
''')

report: GapReport = (prompt | structured).invoke({
    "user_skills": ", ".join(USER_SKILLS),
    "target_role": "ml-engineer",
    "context": retrieved_context,
})

print(f"Target role: {report.target_role}\n")
for i, g in enumerate(report.gaps, 1):
    print(f"{i}. {g.skill}  (rel={g.relevance}, {g.difficulty})")
    print(f"   prereqs: {g.prerequisites}")
    print(f"   why: {g.why}\n")

## 6. LangGraph subgraph

Gap Analysis is one node in the supervisor. Internally it's a small DAG: retrieve → rerank → analyze. This is where the cache check lives in production (Redis key = hash of profile + role).


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class GapState(TypedDict, total=False):
    user_skills: list[str]
    target_role: str
    retrieved: list  # list[(meta, score)]
    report: GapReport

def node_retrieve(state: GapState) -> GapState:
    raw = sorted({t["skill"] for t in TAXONOMY if t["role"] == state["target_role"]} - set(state["user_skills"]))
    query = f"{state['target_role']} role missing: {', '.join(raw)}"
    return {"retrieved": retrieve_for_gap_query(query, role=state["target_role"], top_n=8)}

def node_analyze(state: GapState) -> GapState:
    ctx = "\n".join(
        f"- {m['skill']} (cat={m['skill_category']}, level={m['level_required']}, prereqs={m['prerequisites']}): {m['description']}"
        for m, _ in state["retrieved"]
    )
    report = (prompt | structured).invoke({
        "user_skills": ", ".join(state["user_skills"]),
        "target_role": state["target_role"],
        "context": ctx,
    })
    return {"report": report}

g = StateGraph(GapState)
g.add_node("retrieve", node_retrieve)
g.add_node("analyze", node_analyze)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "analyze")
g.add_edge("analyze", END)
gap_graph = g.compile()

try:
    from IPython.display import Image, display
    display(Image(gap_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(gap_graph.get_graph().draw_mermaid())

In [ ]:
out = gap_graph.invoke({"user_skills": USER_SKILLS, "target_role": "ml-engineer"})
for g in out["report"].gaps:
    print(f"- {g.skill:20s} rel={g.relevance:3d} {g.difficulty:7s} prereqs={g.prerequisites}")

## 7. Notes for the next engineer

- **Pinecone swap.** Replace the `index.add(vectors)` block with `pinecone.upsert` and pass `filter={"role": target_role, "corpus": "taxonomy"}` to `.query()`. The metadata pre-filter is a Pinecone native feature — no Python filtering needed.
- **Namespaces.** Docs call for three corpora under separate Pinecone namespaces within the single free-tier index: `taxonomy`, `course_catalog`, `jd_corpus`. Keep each record as a single chunk (don't sub-chunk courses — it breaks the skill ↔ prereq linkage).
- **BM25 in Pinecone-land.** Pinecone free tier is one index only. Keep BM25 as a Python `rank_bm25` pre-filter over the metadata-pre-filtered subset (usually < 500 nodes per role, so it's cheap).
- **Weights.** `BM25=0.4 / dense=0.6` for skill-name lookups. `BM25=0.2 / dense=0.8` for JD corpus lookups. Expose as a retriever kwarg.
- **Reranker model.** `cross-encoder/ms-marco-MiniLM-L-6-v2` is the sweet spot (~50ms/pair on CPU). Cache it at process start.
- **Prerequisite graph.** Traverse prereqs in Python, NOT via RAG. We already store them as structured metadata. That's also what feeds the Roadmap Generation downstream.
- **Downstream.** The `GapReport` output is the input to `03_deep_researcher.ipynb`.
